# Explore shot selections

Visually compare the per-pixel feature image produced by different `ShotSelection`
recipes, using `automask.viz`. A feature = a `reduction` (`mean`/`std`) over the
shots a `ShotSelection` keeps; see `automask/shot_selection.py` for the knobs
(`beam`, `cc`, `vcc`, `n_shots`, `filter_low`/`filter_high`, `intensity`,
`normalization`).

**Physics.** There is no laser in this experiment: one x-ray beam is split into
the **CC** and **VCC** branches, each with its own shutter (`ai/ch02`/`ai/ch03`
thresholded at 2 V). `beam` is a separate axis — EVR code 137, "did the machine
deliver x-rays at all". See the repo-root `DATA.md`.

**Branch availability.** CC is open on 100% of shots in both local runs, so
`cc='open'` is free. VCC is open on 90% of run 389 but **0% of run 475** — a
`vcc='open'` selection on run 475 raises rather than returning nothing.

**Kernel.** Select the **`Python (ana-psana)`** kernel (top-right). It's the
`ana-4.0.62` conda env — the only one with both `automask` *and* psana, and it
has the `SIT_*` data vars baked in, so both cached and uncached selections work.
Any kernel lacking `automask` fails at the import cell; one with `automask` but
no psana can only render selections already in the cache.

**Cache note.** A selection already in the FeatureStore cache renders instantly
(numpy only). A *new* selection triggers a raw-XTC pass (needs the psana kernel
above; first render is slow, then it's cached). The CC/VCC rework invalidated
every pre-existing cache entry, so expect XTC passes until you re-prewarm.

Present runs: **389** and **475**.

In [ ]:
import os
os.environ.setdefault("SIT_PSDM_DATA", "/Data/hippolyte.wallaert/psdm")
os.environ.setdefault("SIT_ROOT", "/Data/hippolyte.wallaert/psdm/sit_root")
os.environ.setdefault("SIT_DATA", "/Data/hippolyte.wallaert/psdm/data")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from automask import viz
from automask.shot_selection import ShotSelection

RUN = 389  # or 475

# Selections prewarmed by `python -m automask.producers.build_features`:
#   mean: beam=on (the _LIT default), beam=off (the dark)
#   std : beam=on

## 1. A single feature

`show_feature` accepts a catalogue name (`'umean'`, `'ustd'`, `'mean'`), a
`FeatureSpec`, or a bare `ShotSelection` (paired with `reduction`, default
`'mean'`). Dead/zero pixels are shown neutral; scale is a robust 1–99th pct.

In [ ]:
from automask.features import FeatureSpec

reductions = ["mean", "std", "median", "mad"]

selection = ShotSelection(
    beam="on",
    cc="open",
    vcc="any",
    n_shots=100,
    filter_low=0.03,
    filter_high=0.03,
    intensity="sample_diode",
    normalization="none"
)

feature_spec = FeatureSpec(
    name="on",
    selection=selection,
    reduction="mean"
)

In [ ]:
viz.show_feature(RUN, feature_spec)   # beam-on per-pixel mean
plt.show()

In [ ]:
viz.show_feature(RUN, 'mean')   # beam-off dark frame
plt.show()

In [ ]:
viz.show_feature(RUN, 'ustd')   # beam-on per-pixel std
plt.show()

## 2. Effect of a shot selector: beam-on vs i0-normalized vs dark

`compare_selections` renders one panel per selection on a **shared** color scale
and colorbar, so brightness differences between recipes are real (not per-panel
autoscaled). Here: plain beam-on mean vs the same with per-shot `sample_diode`
normalization vs the beam-off dark.

Note `sample_diode` (the lab's own normalizer) is *downstream* of the CC/VCC
split, so unlike ipm2 it tracks the flux actually reaching the detector.

In [ ]:
viz.compare_selections(
    RUN,
    [
        ShotSelection(beam='on'),                                   # beam-on mean
        ShotSelection(beam='on', normalization='sample_diode'),     # + per-shot i0 scaling
        ShotSelection(beam='off'),                                  # beam-off dark
    ],
    reduction='mean',
)
plt.show()

## 3. Same idea on the std reduction

The beam-on per-pixel std (`ustd`) is where scattering contrast lives. Compare
plain vs `sample_diode`-normalized.

In [ ]:
viz.compare_selections(
    RUN,
    [
        ShotSelection(beam='on'),
        ShotSelection(beam='on', normalization='sample_diode'),
    ],
    reduction='std',
)
plt.show()

## Investigate from here

Swap in your own selections below. Any combination of the `ShotSelection` knobs
works; remember an **uncached** recipe needs the psana env (it does a raw-XTC
pass on first render, then caches). Ideas to try:

- intensity trim: `filter_low` / `filter_high` (e.g. `0.0` vs `0.1`) — how much
  do the brightest/dimmest shots move the mean?
- the branch state: `vcc='open'` vs `'closed'` on **run 389** (run 475 has no
  VCC-open shots). This is the real physical knob — the two branches deposit
  visibly different flux on the detector.
- `intensity=` — compare a downstream monitor (`sample_diode`, `diodeU`,
  `lombpm`) against the upstream `ipm2`, which is blind to the branch state.
- `reduction='std'` vs `'mean'`.

`show`, `show_feature`, and `compare_selections` all take an optional `ax=` and
return their artist/figure — pass `out='foo.png'` to save.

In [ ]:
viz.compare_selections(
    389,   # run 475 has zero VCC-open shots
    [
        ShotSelection(beam="on", vcc="open", n_shots=800),
        ShotSelection(beam="on", vcc="closed", n_shots=800),
    ],
    reduction='mean',
)
plt.show()